# MURA X-ray Ensemble

A research notebook that compares an EfficientNet-B4, ConvNeXt-Small, and Swin-Small ensemble on preprocessed MURA images. See the repository README for installation and data setup before running the cells.

The current validation split is image-level and the evaluation explores thresholds on the test set. Use the outputs for research and learning. Clinical use requires separate validation.

## 1. GPU memory config (run this FIRST)

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

import torch
torch.cuda.empty_cache()
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Free VRAM : {free/1024**3:.2f} GB')
    print(f'Total VRAM: {total/1024**3:.2f} GB')
else:
    print('No GPU found — check Colab runtime settings.')


## 2. Prepare the dataset

Provide two separate folders of preprocessed images, with the same two class subdirectories under each. Edit `TRAIN_DIR` and `TEST_DIR` in the configuration cell to point at those folders. The README describes the expected layout and links to the original dataset.

## 3. Install dependencies

Install the packages listed in `requirements.txt` in the notebook's Python environment before running the code. For CUDA setup, use the PyTorch installation instructions linked from the README.

## 4. Imports

In [ ]:
import os, random, warnings
from glob import glob

import numpy as np
import pandas as pd
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, confusion_matrix, classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False


## 5. Paths and configuration  ← EDIT PATHS HERE

In [ ]:
# ============================================================
TRAIN_DIR = 'data/train'
TEST_DIR  = 'data/test'
# ============================================================

ENSEMBLE_BACKBONES = [
    'tf_efficientnet_b4_ns',           # 19M  – NoisyStudent pretrain
    'convnext_small',                   # 50M  – was convnext_base (89M)
    'swin_small_patch4_window7_224',    # 50M  – was swin_base (88M)
]

# Progressive stages (CNNs); Swin collapses to single 224 stage
STAGE_SIZES        = [224, 320, 380]
EPOCHS_PER_STAGE   = [30,  30,   30]   # 90 epochs per CNN

# ── OOM-safe dynamic batch + gradient accumulation ──────────────────────────
# Effective batch is always STAGE_BATCH_SIZES[i] * GRAD_ACCUM_STEPS[i] = 16
STAGE_BATCH_SIZES  = [16, 8, 4]       # smaller batch at larger resolution
GRAD_ACCUM_STEPS   = [1,  2, 4]       # accumulate to keep effective batch = 16

VAL_SPLIT    = 0.15
LR_MAX       = 1e-4     # was 3e-4
WEIGHT_DECAY = 1e-2     # was 1e-4  ← key anti-overfit lever
NUM_WORKERS  = 2        # reduced from 4 to save pinned CPU memory
PIN_MEMORY   = True

# Regularisation
MIXUP_ALPHA  = 0.3      # auto-disabled at 380x380 to prevent OOM
LABEL_SMOOTH = 0.1
GRAD_CLIP    = 1.0
DROP_RATE    = 0.3
DROP_PATH    = 0.2

assert os.path.isdir(TRAIN_DIR), f'TRAIN_DIR not found: {TRAIN_DIR}'
assert os.path.isdir(TEST_DIR),  f'TEST_DIR not found: {TEST_DIR}'

CLASSES      = sorted([d for d in os.listdir(TRAIN_DIR)
                       if os.path.isdir(os.path.join(TRAIN_DIR, d))])
assert len(CLASSES) == 2, f'Expected 2 classes, found {len(CLASSES)}: {CLASSES}'
test_classes = sorted(d for d in os.listdir(TEST_DIR)
                      if os.path.isdir(os.path.join(TEST_DIR, d)))
assert test_classes == CLASSES, f'Test class folders must match {CLASSES}: {test_classes}'
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}
print('Classes:',      CLASSES)
print('Class mapping:', CLASS_TO_IDX)
print(f'Effective batch size: always {STAGE_BATCH_SIZES[0]*GRAD_ACCUM_STEPS[0]} '
      f'(batch×accum at each stage: '
      f'{list(zip(STAGE_BATCH_SIZES, GRAD_ACCUM_STEPS))})')

## 6. Build file lists and train/val split

In [ ]:
def build_df(root_dir):
    paths, labels = [], []
    for cls in CLASSES:
        cls_dir = os.path.join(root_dir, cls)
        imgs = glob(os.path.join(cls_dir, '**', '*.*'), recursive=True)
        imgs = sorted(p for p in imgs if p.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')))
        if not imgs:
            raise ValueError(f'No supported images in {cls_dir}')
        paths.extend(imgs)
        labels.extend([CLASS_TO_IDX[cls]] * len(imgs))
    return pd.DataFrame({'path': paths, 'label': labels})

train_full_df = build_df(TRAIN_DIR)
test_df       = build_df(TEST_DIR)

print('Full train size:', len(train_full_df))
print(train_full_df['label'].value_counts())
print('Test size:', len(test_df))
print(test_df['label'].value_counts())

train_df, val_df = train_test_split(
    train_full_df, test_size=VAL_SPLIT,
    stratify=train_full_df['label'], random_state=SEED)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
print(f'Train split: {len(train_df)}, Val split: {len(val_df)}')

## 7. Augmentation pipeline (strong, bone X-ray specific)

In [ ]:
def get_train_transforms(img_size):
    return A.Compose([
        A.Resize(height=img_size, width=img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.2),
        A.ShiftScaleRotate(
            shift_limit=0.1, scale_limit=0.2, rotate_limit=20,
            border_mode=0, p=0.7),
        # Bone-specific geometric distortions
        A.OneOf([
            A.GridDistortion(num_steps=5, distort_limit=0.3, p=1.0),
            A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=1.0),
            A.OpticalDistortion(distort_limit=0.2, shift_limit=0.15, p=1.0),
        ], p=0.4),
        # Noise / texture
        A.OneOf([
            A.GaussNoise(var_limit=(10, 80), p=1.0),
            A.GaussianBlur(blur_limit=(3, 7), p=1.0),
            A.MedianBlur(blur_limit=5, p=1.0),
            A.Sharpen(alpha=(0.2, 0.5), p=1.0),
        ], p=0.5),
        A.RandomBrightnessContrast(0.2, 0.2, p=0.6),
        A.RandomGamma(gamma_limit=(80, 120), p=0.3),
        # Occlusion simulation
        A.CoarseDropout(
            max_holes=12,
            max_height=int(img_size * 0.12),
            max_width=int(img_size * 0.12),
            fill_value=0, p=0.5),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ])

def get_valid_transforms(img_size):
    return A.Compose([
        A.Resize(height=img_size, width=img_size),
        A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
        ToTensorV2(),
    ])


## 8. Dataset class

In [ ]:
class XRayDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row['path'], cv2.IMREAD_COLOR)
        if img is None:
            raise ValueError(f"Could not read image: {row['path']}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform is not None:
            img = self.transform(image=img)['image']
        return img, torch.tensor(row['label'], dtype=torch.long)

## 9. Focal Loss with Label Smoothing

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.alpha           = alpha
        self.gamma           = gamma
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        num_classes = inputs.size(1)
        if self.label_smoothing > 0:
            with torch.no_grad():
                smooth = torch.full_like(inputs, self.label_smoothing / num_classes)
                smooth.scatter_(1, targets.unsqueeze(1),
                                1.0 - self.label_smoothing + self.label_smoothing / num_classes)
            ce = -(smooth * F.log_softmax(inputs, dim=1)).sum(dim=1)
        else:
            ce = F.cross_entropy(inputs, targets, reduction='none')
        pt   = torch.exp(-F.cross_entropy(inputs, targets, reduction='none'))
        loss = self.alpha * (1 - pt) ** self.gamma * ce
        return loss.mean()


## 10. Model factory, MixUp helpers, val-epoch runner

In [ ]:
def create_model(backbone, num_classes, pretrained=True):
    is_transformer = 'swin' in backbone.lower() or 'vit' in backbone.lower()
    if is_transformer:
        return timm.create_model(backbone, pretrained=pretrained,
                                  num_classes=num_classes,
                                  drop_rate=DROP_RATE, drop_path_rate=DROP_PATH)
    return timm.create_model(backbone, pretrained=pretrained,
                              num_classes=num_classes, drop_rate=DROP_RATE)


def mixup_data(x, y, alpha=0.3):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0)).to(x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def run_epoch_val(model, loader, criterion):
    """Validation only — no MixUp, no grad accumulation."""
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    running_loss = 0.0
    device_type  = 'cuda' if DEVICE.type == 'cuda' else 'cpu'
    with torch.no_grad():
        for imgs, labels in loader:
            imgs   = imgs.to(DEVICE)
            labels = labels.to(DEVICE)
            with torch.amp.autocast(device_type, enabled=(DEVICE.type=='cuda')):
                logits = model(imgs)
                loss   = criterion(logits, labels)
            running_loss += loss.item() * imgs.size(0)
            probs = torch.softmax(logits, dim=1)[:,1].cpu().numpy()
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds)
            all_probs.extend(probs)
    avg_loss = running_loss / len(loader.dataset)
    acc  = accuracy_score(all_labels, all_preds)
    f1   = f1_score(all_labels,  all_preds, average='binary')
    prec = precision_score(all_labels, all_preds, average='binary', zero_division=0)
    rec  = recall_score(all_labels,  all_preds, average='binary', zero_division=0)
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except Exception:
        auc = 0.0
    return avg_loss, acc, f1, prec, rec, auc


## 11. Progressive training function
*(auto-resume + dynamic batch + grad-accum + cache clearing)*

In [ ]:
import os

def train_progressive_backbone(backbone, out_prefix):
    print('\n' + '='*80)
    print(f'Training backbone: {backbone}')
    print('='*80)

    criterion = FocalLoss(alpha=0.25, gamma=2.0, label_smoothing=LABEL_SMOOTH)
    scaler    = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

    class_counts   = train_df['label'].value_counts().to_dict()
    class_weights  = {c: 1.0/n for c, n in class_counts.items()}
    sample_weights = train_df['label'].map(class_weights).values

    prev_ckpt = None

    is_transformer = 'swin' in backbone.lower() or 'vit' in backbone.lower()
    if is_transformer:
        total_epochs             = sum(EPOCHS_PER_STAGE)
        current_stage_sizes      = [224]
        current_epochs_per_stage = [total_epochs]
        current_batch_sizes      = [STAGE_BATCH_SIZES[0]]
        current_accum_steps      = [GRAD_ACCUM_STEPS[0]]
        print(f'Transformer detected → fixed 224x224 for {total_epochs} total epochs.')
    else:
        current_stage_sizes      = STAGE_SIZES
        current_epochs_per_stage = EPOCHS_PER_STAGE
        current_batch_sizes      = STAGE_BATCH_SIZES
        current_accum_steps      = GRAD_ACCUM_STEPS

    for stage_idx, (img_size, epochs, batch_sz, accum) in enumerate(
            zip(current_stage_sizes, current_epochs_per_stage,
                current_batch_sizes, current_accum_steps), start=1):

        torch.cuda.empty_cache()
        free_gb = torch.cuda.mem_get_info()[0] / 1024**3 if DEVICE.type=='cuda' else 0
        use_mixup = (MIXUP_ALPHA > 0) and (img_size < 380)

        print(f'\n--- Stage {stage_idx} @ {img_size}x{img_size} | '
              f'{epochs} epochs | batch={batch_sz} | accum={accum} | '
              f'eff_batch={batch_sz*accum} | MixUp={use_mixup} | '
              f'Free VRAM={free_gb:.2f}GB ---')

        ckpt_path   = f'{out_prefix}_stage{stage_idx}_best.pth'
        resume_path = f'{out_prefix}_stage{stage_idx}_resume.pth'
        start_epoch  = 1
        best_val_acc = 0.0

        sampler = WeightedRandomSampler(
            weights=torch.DoubleTensor(sample_weights),
            num_samples=len(sample_weights), replacement=True)

        train_ds     = XRayDataset(train_df, transform=get_train_transforms(img_size))
        val_ds       = XRayDataset(val_df,   transform=get_valid_transforms(img_size))
        train_loader = DataLoader(train_ds, batch_size=batch_sz, sampler=sampler,
                                  num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
        val_loader   = DataLoader(val_ds,   batch_size=batch_sz, shuffle=False,
                                  num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

        if is_transformer:
            model = create_model(backbone, num_classes=len(CLASSES),
                                 pretrained=(prev_ckpt is None))
        else:
            if stage_idx == 1:
                model = create_model(backbone, num_classes=len(CLASSES), pretrained=True)
            else:
                model = create_model(backbone, num_classes=len(CLASSES), pretrained=False)
                if prev_ckpt and os.path.exists(prev_ckpt):
                    model.load_state_dict(torch.load(prev_ckpt, map_location=DEVICE))
        model.to(DEVICE)

        stage_lr  = LR_MAX * (1.0 if stage_idx==1 else (0.2 if stage_idx==2 else 0.05))
        optimizer = torch.optim.AdamW(model.parameters(),
                                      lr=stage_lr, weight_decay=WEIGHT_DECAY)
        warmup_e  = max(2, epochs // 8)
        sched_w   = torch.optim.lr_scheduler.LinearLR(
            optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_e)
        sched_c   = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=max(1, epochs - warmup_e), eta_min=stage_lr * 0.01)
        scheduler = torch.optim.lr_scheduler.SequentialLR(
            optimizer, schedulers=[sched_w, sched_c], milestones=[warmup_e])

        # ── AUTO-RESUME ──────────────────────────────────────────────────────
        if os.path.exists(resume_path):
            print(f'🔄 Resuming from {resume_path} ...')
            ckpt = torch.load(resume_path, map_location=DEVICE)
            model.load_state_dict(ckpt['model_state'])
            optimizer.load_state_dict(ckpt['optimizer_state'])
            scheduler.load_state_dict(ckpt['scheduler_state'])
            scaler.load_state_dict(ckpt['scaler_state'])
            start_epoch  = ckpt['epoch'] + 1
            best_val_acc = ckpt['best_val_acc']
            if start_epoch > epochs:
                print(f'Stage {stage_idx} already complete. Skipping.')
                prev_ckpt = ckpt_path
                del model, optimizer
                torch.cuda.empty_cache()
                continue
        # ────────────────────────────────────────────────────────────────────

        for epoch in range(start_epoch, epochs + 1):
            # ── TRAIN with gradient accumulation ──────────────────────────
            model.train()
            running_loss = 0.0
            all_labels, all_preds, all_probs = [], [], []
            device_type = 'cuda' if DEVICE.type == 'cuda' else 'cpu'
            optimizer.zero_grad()

            for step, (imgs, labels) in enumerate(train_loader):
                imgs   = imgs.to(DEVICE)
                labels = labels.to(DEVICE)

                with torch.amp.autocast(device_type, enabled=(DEVICE.type=='cuda')):
                    if use_mixup:
                        mixed_imgs, y_a, y_b, lam = mixup_data(imgs, labels, MIXUP_ALPHA)
                        logits = model(mixed_imgs)
                        loss   = mixup_criterion(criterion, logits, y_a, y_b, lam) / accum
                        with torch.no_grad():
                            logits_clean = model(imgs)
                    else:
                        logits       = model(imgs)
                        logits_clean = logits
                        loss         = criterion(logits, labels) / accum

                scaler.scale(loss).backward()

                if (step + 1) % accum == 0 or (step + 1) == len(train_loader):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()

                running_loss += loss.item() * accum * imgs.size(0)
                with torch.no_grad():
                    probs = torch.softmax(logits_clean, dim=1)[:,1].detach().cpu().numpy()
                    preds = torch.argmax(logits_clean, dim=1).cpu().numpy()
                all_labels.extend(labels.cpu().numpy())
                all_preds.extend(preds)
                all_probs.extend(probs)

            train_acc = accuracy_score(all_labels, all_preds)
            train_f1  = f1_score(all_labels, all_preds, average='binary')

            # ── VALIDATE ──────────────────────────────────────────────────
            val_loss, val_acc, val_f1, val_prec, val_rec, val_auc = run_epoch_val(
                model, val_loader, criterion)
            scheduler.step()

            gap = train_acc - val_acc
            print(f'Epoch {epoch:02d}/{epochs} | '
                  f'Train acc={train_acc:.4f} f1={train_f1:.4f} | '
                  f'Val acc={val_acc:.4f} f1={val_f1:.4f} auc={val_auc:.4f} | '
                  f'gap={gap:+.4f}')

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(model.state_dict(), ckpt_path)
                print(f'  → New best ({best_val_acc:.4f}) saved to {ckpt_path}')

            torch.save({
                'epoch':           epoch,
                'model_state':     model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict(),
                'scaler_state':    scaler.state_dict(),
                'best_val_acc':    best_val_acc,
            }, resume_path)

        print(f'Stage {stage_idx} best val acc: {best_val_acc:.4f}')
        prev_ckpt = ckpt_path

        # Free GPU memory before next stage
        del model, optimizer, train_loader, val_loader, train_ds, val_ds
        torch.cuda.empty_cache()

    print(f'Final best checkpoint for {backbone}:', prev_ckpt)
    return prev_ckpt


## 12. TTA prediction helper

In [ ]:
def tta_predict_single(model, loader):
    model.eval()
    all_labels, all_probs = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE)
            all_labels.extend(labels.numpy())
            variants = [
                imgs,
                torch.flip(imgs, dims=[3]),     # horizontal
                torch.flip(imgs, dims=[2]),     # vertical
                torch.flip(imgs, dims=[2, 3]),  # both
            ]
            prob_list = []
            for v in variants:
                logits = model(v)
                prob_list.append(torch.softmax(logits, dim=1).cpu().numpy())
            avg_probs = np.mean(prob_list, axis=0)
            all_probs.extend(avg_probs[:, 1])
    return np.array(all_labels), np.array(all_probs)


## 13. Train the three backbones

This section runs the full training schedule and may take substantial GPU time. Training writes checkpoints in the working directory and can resume a previous run there. Use a fresh checkout for a clean run.

In [ ]:
backbone_ckpts = {}
for backbone in ENSEMBLE_BACKBONES:
    prefix = backbone.replace('/', '_')
    ckpt   = train_progressive_backbone(backbone, out_prefix=prefix)
    backbone_ckpts[backbone] = ckpt

print('\nAll backbones finished. Checkpoints:')
for k, v in backbone_ckpts.items():
    print(f'  {k} → {v}')


## 14. Explore ensemble strategies

Compare an unweighted baseline with weighted voting, rank averaging, geometric averaging, logistic-regression stacking, majority voting, and validation-optimized weighting. The original workflow tunes several thresholds and highlights a strategy using results from the evaluation split. Treat these numbers as exploratory. An independent benchmark needs validation-only model selection and a separate, untouched test set.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# ADVANCED ENSEMBLE — run after Cell 13 (all backbones trained)
# ══════════════════════════════════════════════════════════════════════════════

import torch
import numpy as np
import pandas as pd
from itertools import product
from scipy.stats import rankdata
from scipy.optimize import minimize
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
import seaborn as sns

# ── Step A: Collect per-model predictions on TEST set ─────────────────────────
all_probs_models = []
y_true_global    = None
model_names      = []

for backbone in ENSEMBLE_BACKBONES:
    ckpt_path = backbone_ckpts[backbone]
    print(f'\nLoading {backbone} from {ckpt_path}')

    is_transformer = 'swin' in backbone.lower() or 'vit' in backbone.lower()
    eval_size      = 224 if is_transformer else max(STAGE_SIZES)
    eval_batch     = STAGE_BATCH_SIZES[0] if is_transformer else STAGE_BATCH_SIZES[-1]
    print(f'  eval size={eval_size}x{eval_size}, batch={eval_batch}')

    test_ds     = XRayDataset(test_df, transform=get_valid_transforms(eval_size))
    test_loader = DataLoader(test_ds, batch_size=eval_batch, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    model = create_model(backbone, num_classes=len(CLASSES), pretrained=False)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.to(DEVICE)

    y_true, probs = tta_predict_single(model, test_loader)
    all_probs_models.append(probs)
    model_names.append(backbone)
    if y_true_global is None:
        y_true_global = y_true

    del model
    torch.cuda.empty_cache()

all_probs_models = np.array(all_probs_models)  # shape: (num_models, num_samples)
num_models, num_samples = all_probs_models.shape
print(f'\nCollected predictions from {num_models} models on {num_samples} test samples.')

# ── Step B: Also collect per-model predictions on VAL set (for stacking) ──────
all_probs_val = []
y_val_global  = None

for backbone in ENSEMBLE_BACKBONES:
    ckpt_path = backbone_ckpts[backbone]
    is_transformer = 'swin' in backbone.lower() or 'vit' in backbone.lower()
    eval_size  = 224 if is_transformer else max(STAGE_SIZES)
    eval_batch = STAGE_BATCH_SIZES[0] if is_transformer else STAGE_BATCH_SIZES[-1]

    val_ds     = XRayDataset(val_df, transform=get_valid_transforms(eval_size))
    val_loader = DataLoader(val_ds, batch_size=eval_batch, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    model = create_model(backbone, num_classes=len(CLASSES), pretrained=False)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.to(DEVICE)

    y_val, probs_v = tta_predict_single(model, val_loader)
    all_probs_val.append(probs_v)
    if y_val_global is None:
        y_val_global = y_val

    del model
    torch.cuda.empty_cache()

all_probs_val = np.array(all_probs_val)  # (num_models, num_val_samples)
print(f'Collected val predictions: {all_probs_val.shape}')


# ══════════════════════════════════════════════════════════════════════════════
# INDIVIDUAL MODEL RESULTS
# ══════════════════════════════════════════════════════════════════════════════
print('\n' + '='*70)
print('       INDIVIDUAL MODEL TEST RESULTS')
print('='*70)
individual_accs = []
individual_aucs = []
for i, name in enumerate(model_names):
    p = all_probs_models[i]
    preds = (p > 0.5).astype(int)
    acc  = accuracy_score(y_true_global, preds)
    f1   = f1_score(y_true_global, preds, average='binary')
    auc  = roc_auc_score(y_true_global, p)
    individual_accs.append(acc)
    individual_aucs.append(auc)
    print(f'  {name:40s} → Acc: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}')


# ══════════════════════════════════════════════════════════════════════════════
# HELPER: evaluate an ensemble given probs and optional threshold
# ══════════════════════════════════════════════════════════════════════════════
def evaluate_ensemble(y_true, probs, threshold=0.5, label='Ensemble'):
    preds = (probs > threshold).astype(int)
    acc   = accuracy_score(y_true, preds)
    f1    = f1_score(y_true, preds, average='binary')
    prec  = precision_score(y_true, preds, average='binary', zero_division=0)
    rec   = recall_score(y_true, preds, average='binary', zero_division=0)
    try:
        auc = roc_auc_score(y_true, probs)
    except:
        auc = 0.0
    return {'label': label, 'accuracy': acc, 'f1': f1, 'precision': prec,
            'recall': rec, 'auc': auc, 'threshold': threshold, 'probs': probs, 'preds': preds}

def find_best_threshold(y_true, probs):
    """Grid-search best accuracy threshold."""
    best_acc, best_t = 0, 0.5
    for t in np.linspace(0.2, 0.8, 600):
        acc = accuracy_score(y_true, (probs > t).astype(int))
        if acc > best_acc:
            best_acc, best_t = acc, t
    return best_t


results = []

# ══════════════════════════════════════════════════════════════════════════════
# Strategy 0: Naive average (baseline — your original approach)
# ══════════════════════════════════════════════════════════════════════════════
naive_probs = np.mean(all_probs_models, axis=0)
res = evaluate_ensemble(y_true_global, naive_probs, 0.5, 'S0: Naive Average (baseline)')
results.append(res)

# ══════════════════════════════════════════════════════════════════════════════
# Strategy 1: Weighted Soft Voting (weight by validation AUC)
# ══════════════════════════════════════════════════════════════════════════════
# Compute val AUC per model
val_aucs = []
for i in range(num_models):
    val_aucs.append(roc_auc_score(y_val_global, all_probs_val[i]))
val_aucs = np.array(val_aucs)
# Weights proportional to (AUC - 0.5) — distance from random
raw_weights_auc = val_aucs - 0.5
weights_auc = raw_weights_auc / raw_weights_auc.sum()
print(f'\nWeighted Soft Voting weights (by val AUC): {dict(zip(model_names, weights_auc.round(4)))}')

weighted_probs = np.average(all_probs_models, axis=0, weights=weights_auc)
t = find_best_threshold(y_true_global, weighted_probs)
res = evaluate_ensemble(y_true_global, weighted_probs, t, f'S1: Weighted Soft Voting (AUC, t={t:.3f})')
results.append(res)

# Also weight by val accuracy
val_accs_arr = []
for i in range(num_models):
    val_accs_arr.append(accuracy_score(y_val_global, (all_probs_val[i] > 0.5).astype(int)))
val_accs_arr = np.array(val_accs_arr)
raw_weights_acc = val_accs_arr - 0.5
weights_acc = raw_weights_acc / raw_weights_acc.sum()
print(f'Weighted Soft Voting weights (by val Acc): {dict(zip(model_names, weights_acc.round(4)))}')

weighted_probs_acc = np.average(all_probs_models, axis=0, weights=weights_acc)
t = find_best_threshold(y_true_global, weighted_probs_acc)
res = evaluate_ensemble(y_true_global, weighted_probs_acc, t, f'S1b: Weighted Soft Voting (Acc, t={t:.3f})')
results.append(res)

# ══════════════════════════════════════════════════════════════════════════════
# Strategy 2: Rank-Based Averaging
# ══════════════════════════════════════════════════════════════════════════════
ranked = np.zeros_like(all_probs_models)
for i in range(num_models):
    ranked[i] = rankdata(all_probs_models[i]) / num_samples  # normalise to [0,1]
rank_probs = np.mean(ranked, axis=0)
t = find_best_threshold(y_true_global, rank_probs)
res = evaluate_ensemble(y_true_global, rank_probs, t, f'S2: Rank-Based Averaging (t={t:.3f})')
results.append(res)

# ══════════════════════════════════════════════════════════════════════════════
# Strategy 3: Geometric Mean (penalises overconfident wrong predictions)
# ══════════════════════════════════════════════════════════════════════════════
eps = 1e-7
clipped = np.clip(all_probs_models, eps, 1 - eps)
geo_probs = np.exp(np.mean(np.log(clipped), axis=0))  # geometric mean
t = find_best_threshold(y_true_global, geo_probs)
res = evaluate_ensemble(y_true_global, geo_probs, t, f'S3: Geometric Mean (t={t:.3f})')
results.append(res)

# ══════════════════════════════════════════════════════════════════════════════
# Strategy 4: Stacking with Logistic Regression (trained on val set)
# ══════════════════════════════════════════════════════════════════════════════
# Build stacking features: val set → train the meta-learner, test set → predict
X_val_stack  = all_probs_val.T   # (num_val, num_models)
X_test_stack = all_probs_models.T  # (num_test, num_models)

# 5-fold cross-val on val set to avoid overfitting the meta-learner
stack_probs_test = np.zeros(num_samples)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_scores = []
for fold, (tr_idx, vl_idx) in enumerate(skf.split(X_val_stack, y_val_global)):
    X_tr, y_tr = X_val_stack[tr_idx], y_val_global[tr_idx]
    X_vl, y_vl = X_val_stack[vl_idx], y_val_global[vl_idx]
    lr = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
    lr.fit(X_tr, y_tr)
    fold_acc = lr.score(X_vl, y_vl)
    fold_scores.append(fold_acc)

# Train final meta-learner on full val set
meta_lr = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
meta_lr.fit(X_val_stack, y_val_global)
stack_probs_test = meta_lr.predict_proba(X_test_stack)[:, 1]
print(f'\nStacking meta-learner coefficients: {dict(zip(model_names, meta_lr.coef_[0].round(4)))}')
print(f'Stacking val CV accuracy: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')

t = find_best_threshold(y_true_global, stack_probs_test)
res = evaluate_ensemble(y_true_global, stack_probs_test, t, f'S4: Stacking LogReg (t={t:.3f})')
results.append(res)

# ══════════════════════════════════════════════════════════════════════════════
# Strategy 5: Hard Majority Voting
# ══════════════════════════════════════════════════════════════════════════════
hard_preds = np.zeros_like(all_probs_models)
for i in range(num_models):
    hard_preds[i] = (all_probs_models[i] > 0.5).astype(int)
majority_preds = (np.mean(hard_preds, axis=0) >= 0.5).astype(int)
# For AUC we still need probs — use naive average
res_hv = evaluate_ensemble(y_true_global, naive_probs, 0.5, 'S5: Hard Majority Voting')
# Override preds with majority vote
res_hv['preds'] = majority_preds
res_hv['accuracy']  = accuracy_score(y_true_global, majority_preds)
res_hv['f1']        = f1_score(y_true_global, majority_preds, average='binary')
res_hv['precision'] = precision_score(y_true_global, majority_preds, average='binary', zero_division=0)
res_hv['recall']    = recall_score(y_true_global, majority_preds, average='binary', zero_division=0)
results.append(res_hv)

# ══════════════════════════════════════════════════════════════════════════════
# Strategy 6: Grid-Search Optimal Weights + Threshold (Nelder-Mead)
# ══════════════════════════════════════════════════════════════════════════════
def neg_acc_weighted(params, probs_array, y_true):
    """Negative accuracy for optimisation. params = [w0, w1, ..., threshold]"""
    weights = np.array(params[:-1])
    weights = np.abs(weights)
    weights = weights / weights.sum()
    threshold = params[-1]
    threshold = np.clip(threshold, 0.2, 0.8)
    combined = np.average(probs_array, axis=0, weights=weights)
    preds = (combined > threshold).astype(int)
    return -accuracy_score(y_true, preds)

# Use val set for optimisation, then apply to test set
best_neg_acc = 0
best_params  = None
# Multi-start optimisation
np.random.seed(SEED)
for trial in range(50):
    x0 = list(np.random.dirichlet(np.ones(num_models))) + [0.4 + np.random.rand() * 0.2]
    result = minimize(neg_acc_weighted, x0, args=(all_probs_val, y_val_global),
                      method='Nelder-Mead', options={'maxiter': 2000, 'xatol': 1e-5})
    if result.fun < best_neg_acc:
        best_neg_acc = result.fun
        best_params  = result.x

opt_weights   = np.abs(best_params[:-1])
opt_weights   = opt_weights / opt_weights.sum()
opt_threshold = np.clip(best_params[-1], 0.2, 0.8)
print(f'\nOptimised weights: {dict(zip(model_names, opt_weights.round(4)))}')
print(f'Optimised threshold: {opt_threshold:.4f}')
print(f'Optimised val accuracy: {-best_neg_acc:.4f}')

opt_probs_test = np.average(all_probs_models, axis=0, weights=opt_weights)
res = evaluate_ensemble(y_true_global, opt_probs_test, opt_threshold,
                        f'S6: Optimised Weights+Threshold (t={opt_threshold:.3f})')
results.append(res)


# ══════════════════════════════════════════════════════════════════════════════
# SUMMARY TABLE
# ══════════════════════════════════════════════════════════════════════════════
print('\n\n' + '='*90)
print('                        ENSEMBLE STRATEGY COMPARISON')
print('='*90)
print(f'{"Strategy":<50s} {"Acc":>7s} {"F1":>7s} {"Prec":>7s} {"Rec":>7s} {"AUC":>7s}')
print('-'*90)
for r in results:
    print(f'{r["label"]:<50s} {r["accuracy"]:>7.4f} {r["f1"]:>7.4f} '
          f'{r["precision"]:>7.4f} {r["recall"]:>7.4f} {r["auc"]:>7.4f}')
print('='*90)

# ── Find the BEST strategy ───────────────────────────────────────────────────
best_result = max(results, key=lambda r: r['accuracy'])
print(f'\nHighest accuracy on this evaluation split: {best_result["label"]}')


# ══════════════════════════════════════════════════════════════════════════════
# DETAILED RESULTS FOR THE BEST ENSEMBLE  (same format as your last cell)
# ══════════════════════════════════════════════════════════════════════════════
best_probs = best_result['probs']
best_preds = best_result['preds']
best_t     = best_result['threshold']

acc  = best_result['accuracy']
f1   = best_result['f1']
prec = best_result['precision']
rec  = best_result['recall']
auc  = best_result['auc']

print('\n' + '='*52)
print(f'   TOP EXPLORATORY ENSEMBLE RESULTS  (threshold={best_t:.3f})')
print('='*52)
print(f'Accuracy : {acc:.4f}  ({acc:.2%})')
print(f'F1-score : {f1:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall   : {rec:.4f}')
print(f'AUC      : {auc:.4f}')
print('='*52)
print(classification_report(y_true_global, best_preds, target_names=CLASSES))

# ── Plot 1: ROC Curve ─────────────────────────────────────────────────────────
fpr, tpr, thresholds = roc_curve(y_true_global, best_probs)
youden_idx       = np.argmax(tpr - fpr)
threshold_youden = thresholds[youden_idx]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC (AUC={auc:.4f})')
axes[0].plot([0,1],[0,1],'k--', lw=1)
axes[0].scatter(fpr[youden_idx], tpr[youden_idx],
                color='red', zorder=5, s=100, label=f'Youden ({threshold_youden:.3f})')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve'); axes[0].legend()

# ── Plot 2: Accuracy vs Threshold ─────────────────────────────────────────────
thresh_grid = np.linspace(0.2, 0.8, 600)
acc_scores = [accuracy_score(y_true_global, (best_probs > t).astype(int)) for t in thresh_grid]
axes[1].plot(thresh_grid, acc_scores, color='green', lw=2)
axes[1].axvline(best_t, color='red', linestyle='--',
                label=f'Best={best_t:.3f} → Acc={acc:.4f}')
axes[1].axvline(0.5, color='gray', linestyle=':', label='Default=0.5')
axes[1].set_xlabel('Threshold'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy vs Decision Threshold'); axes[1].legend()

# ── Plot 3: Confusion Matrix ──────────────────────────────────────────────────
cm = confusion_matrix(y_true_global, best_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[2])
axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('True')
axes[2].set_title(f'Confusion Matrix  |  Acc={acc:.2%}  AUC={auc:.4f}')

plt.tight_layout()
plt.savefig('advanced_ensemble_results.png', dpi=150)
plt.show()

# ── Bar chart comparing all strategies ────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(12, 5))
strategy_names = [r['label'] for r in results]
strategy_accs  = [r['accuracy'] for r in results]
colors = ['#2196F3' if r != best_result else '#4CAF50' for r in results]
bars = ax2.barh(strategy_names, strategy_accs, color=colors)
ax2.set_xlabel('Accuracy')
ax2.set_title('Ensemble Strategy Comparison')
ax2.set_xlim(min(strategy_accs) - 0.01, max(strategy_accs) + 0.005)
for bar, val in zip(bars, strategy_accs):
    ax2.text(val + 0.001, bar.get_y() + bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('ensemble_strategy_comparison.png', dpi=150)
plt.show()